# Rolling-Origin Passenger Demand Backtest

This notebook evaluates the selected `3_combined` XGBoost feature set on the latest 24 one-month-ahead origins. At each origin, the training cutoff advances by one month. It compares XGBoost with a same-route 12-month seasonal-naive baseline on common route-months and reports overall and meteorological-season results.

The feature definitions and data checks are maintained in [`modeling.ipynb`](modeling.ipynb); the first code cell runs those preparation cells so this notebook can be executed independently.

In [6]:
import json
from pathlib import Path

# Reuse the canonical data and feature preparation without duplicating it.
data_dir_candidates = [
    Path.cwd() / "team-06-modeling",
    Path.cwd(),
    Path.cwd().parent,
]
DATA_DIR = next(
    (
        path
        for path in data_dir_candidates
        if (path / "AA_PAX_FORECASTING.csv").is_file()
        and (path / "AA_AIRPORT_INFO.csv").is_file()
    ),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find both CSV files. Place them in team-06-modeling/ "
        "or update DATA_DIR to their folder."
    )

modeling_path = DATA_DIR / "notebooks" / "modeling.ipynb"
modeling_notebook = json.loads(modeling_path.read_text(encoding="utf-8"))
for cell_number, cell in enumerate(modeling_notebook["cells"], start=1):
    if cell["cell_type"] == "code":
        source = "".join(cell["source"])
        exec(compile(source, f"{modeling_path} cell {cell_number}", "exec"), globals())

Passenger records: 1,533,522
Date range: 2019-01 to 2026-07
Directional routes: 17,119
Airport reference records: 4,565
1A_route_calendar: 8 predictors
1B_route_calendar_history: 21 predictors
2_route_calendar_geography: 23 predictors
3_combined: 36 predictors
Airport metadata missing for origin: 0.0%
Airport metadata missing for destination: 0.0%
Feature tables are ready; for example, use get_feature_frame('1A_route_calendar').


In [7]:
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, str(DATA_DIR / "models"))
from forecasting import _fit_and_forecast_xgb, _score

feature_set_name = "3_combined"
route_columns = [ORIGIN, DESTINATION]
seasonal_lag = "route_pax_lag_12m"
if seasonal_lag not in features:
    raise ValueError(f"Features must include {seasonal_lag!r}.")

latest_month = passengers[MONTH].max()
target_months = pd.date_range(
    latest_month - pd.DateOffset(months=23), latest_month, freq="MS"
)
observed_months = pd.DatetimeIndex(passengers[MONTH].drop_duplicates())
if not target_months.isin(observed_months).all():
    raise ValueError("The latest 24 calendar months must all have observations.")

season_by_month = {
    12: "DJF", 1: "DJF", 2: "DJF",
    3: "MAM", 4: "MAM", 5: "MAM",
    6: "JJA", 7: "JJA", 8: "JJA",
    9: "SON", 10: "SON", 11: "SON",
}
fold_predictions = []
monthly_metrics = []

for forecast_month in target_months:
    # Only months up to this cutoff are available to fit the model.
    train_end = forecast_month - pd.DateOffset(months=1)
    xgb_predictions = _fit_and_forecast_xgb(
        features, passengers, FEATURE_SETS, feature_set_name, route_columns,
        MONTH, MONTH_INDEX, TARGET, train_end, 1,
    )
    actual = passengers.loc[
        passengers[MONTH] == forecast_month,
        route_columns + [MONTH, TARGET],
    ].copy()
    actual["HORIZON"] = 1
    baseline = features.loc[
        (features[MONTH] == forecast_month) & features[seasonal_lag].notna(),
        route_columns + [MONTH, seasonal_lag],
    ].rename(columns={seasonal_lag: "SEASONAL_NAIVE_PRED"})
    scored = actual.merge(
        xgb_predictions,
        on=route_columns + [MONTH, "HORIZON"],
        how="inner",
        validate="one_to_one",
    ).merge(
        baseline,
        on=route_columns + [MONTH],
        how="inner",
        validate="one_to_one",
    )
    if scored.empty:
        raise ValueError(f"No common route-month predictions for {forecast_month:%Y-%m}.")
    actual_volume = actual[TARGET].sum()
    if actual_volume <= 0:
        raise ValueError(f"Actual passenger volume is not positive for {forecast_month:%Y-%m}.")
    scored["SEASON"] = scored[MONTH].dt.month.map(season_by_month)
    fold_predictions.append(
        scored[route_columns + [MONTH, TARGET, "XGB_PRED", "SEASONAL_NAIVE_PRED", "SEASON"]]
    )

    for model_name, prediction_column in (
        ("XGBoost", "XGB_PRED"),
        ("Seasonal naive", "SEASONAL_NAIVE_PRED"),
    ):
        monthly_metrics.append(
            {
                "Model": model_name,
                "Forecast_Month": forecast_month,
                "Training_Through": train_end,
                "Scored_Route_Months": len(scored),
                "Scored_Routes": scored[route_columns].drop_duplicates().shape[0],
                "Passenger_Volume_Coverage": scored[TARGET].sum() / actual_volume,
                **_score(scored, prediction_column, TARGET),
            }
        )

all_scored = pd.concat(fold_predictions, ignore_index=True)
all_actual = passengers.loc[passengers[MONTH].isin(target_months)]
seasonal_metrics = []
for season in ("ALL", "DJF", "MAM", "JJA", "SON"):
    if season == "ALL":
        scored = all_scored
        actual = all_actual
    else:
        scored = all_scored.loc[all_scored["SEASON"] == season]
        actual = all_actual.loc[all_actual[MONTH].dt.month.isin(
            {"DJF": (12, 1, 2), "MAM": (3, 4, 5), "JJA": (6, 7, 8), "SON": (9, 10, 11)}[season]
        )]
    actual_volume = actual[TARGET].sum()
    if scored.empty or actual_volume <= 0:
        raise ValueError(f"Insufficient actual or scored data for {season}.")
    for model_name, prediction_column in (
        ("XGBoost", "XGB_PRED"),
        ("Seasonal naive", "SEASONAL_NAIVE_PRED"),
    ):
        seasonal_metrics.append(
            {
                "Model": model_name,
                "Season": season,
                "Forecast_Months": scored[MONTH].nunique(),
                "Scored_Route_Months": len(scored),
                "Scored_Routes": scored[route_columns].drop_duplicates().shape[0],
                "Passenger_Volume_Coverage": scored[TARGET].sum() / actual_volume,
                **_score(scored, prediction_column, TARGET),
            }
        )

monthly_metrics = pd.DataFrame(monthly_metrics)
seasonal_metrics = pd.DataFrame(seasonal_metrics)
output_dir = DATA_DIR.parent / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
monthly_path = output_dir / "rolling_origin_monthly_metrics.csv"
seasonal_path = output_dir / "rolling_origin_seasonal_metrics.csv"
monthly_metrics.to_csv(monthly_path, index=False)
seasonal_metrics.to_csv(seasonal_path, index=False)

print(
    f"Rolling one-month-ahead backtest: {target_months.min():%Y-%m} to "
    f"{target_months.max():%Y-%m}; {len(target_months)} expanding origins"
)
print(
    "Common route-month passenger-volume coverage: "
    f"{all_scored[TARGET].sum() / all_actual[TARGET].sum():.1%}"
)
print("\nOverall and meteorological-season metrics:")
print(seasonal_metrics.to_string(index=False, formatters={
    "WAPE": "{:.3f}".format,
    "MAE": "{:.1f}".format,
    "Bias": "{:.3f}".format,
    "Passenger_Volume_Coverage": "{:.1%}".format,
}))
print(f"\nSaved monthly metrics to {monthly_path}")
print(f"Saved seasonal metrics to {seasonal_path}")

Rolling one-month-ahead backtest: 2024-08 to 2026-07; 24 expanding origins
Common route-month passenger-volume coverage: 100.0%

Overall and meteorological-season metrics:
         Model Season  Forecast_Months  Scored_Route_Months  Scored_Routes Passenger_Volume_Coverage  WAPE   MAE   Bias
       XGBoost    ALL               24               410236          17119                    100.0% 0.134 421.7  0.009
Seasonal naive    ALL               24               410236          17119                    100.0% 0.165 519.3  0.037
       XGBoost    DJF                6               102577          17119                    100.0% 0.134 389.2 -0.004
Seasonal naive    DJF                6               102577          17119                    100.0% 0.155 449.2  0.014
       XGBoost    MAM                6               102544          17119                    100.0% 0.138 436.0  0.022
Seasonal naive    MAM                6               102544          17119                    100.0% 0.175 5